# Understanding Boosting — Concept First

I have used Claude to generate this notebook.

---

## What is Boosting?

**One idea, one sentence:**

> Train a series of weak models, where each model focuses on fixing the mistakes of the previous one.

---

### How is this different from Bagging (Random Forest)?

| | Bagging | Boosting |
|---|---|---|
| Trees built | All at once, independently | One by one, each learns from previous errors |
| Data | Random samples | Same data, but mistakes get more attention |
| Goal | Reduce overfit | Reduce errors by focusing on hard cases |
| Final answer | Majority vote | Weighted sum |

Think of it this way:
- **Bagging** = ask 100 people the same question, take the majority answer
- **Boosting** = ask one person, see what they got wrong, ask the next person to specifically fix those, repeat

---
## Our Dataset

6 loan applications. We want to predict: **approved or denied?**

| Row | Income | Loan |
|---|---|---|
| 0 | high | approved |
| 1 | high | approved |
| 2 | low | denied |
| 3 | low | denied |
| 4 | medium | approved |
| 5 | medium | denied |

Row 4 and row 5 are the **tricky ones** — same income, different outcome.  
No single rule can get both right. Boosting will learn to handle this.

In [1]:
import pandas as pd
import math

df = pd.DataFrame([
    {'income': 'high',   'loan': 'approved'},
    {'income': 'high',   'loan': 'approved'},
    {'income': 'low',    'loan': 'denied'},
    {'income': 'low',    'loan': 'denied'},
    {'income': 'medium', 'loan': 'approved'},
    {'income': 'medium', 'loan': 'denied'},
])

df

,income,loan
0,high,approved
1,high,approved
2,low,denied
3,low,denied
4,medium,approved
5,medium,denied


---
# Part 1 — AdaBoost
## "Pay more attention to what you got wrong"

### The idea

Every row starts with an equal **weight** (importance).  
After each round:
- Rows that were **wrong** → weight goes **up** (pay more attention next time)
- Rows that were **correct** → weight goes **down**

Also, each tree gets an **alpha** score — how much to trust it in the final vote.  
A tree that was very accurate gets a high alpha. A weak tree gets a low alpha.

---
### Traced by hand — Round 1

**Initial weights:** all equal → 1/6 ≈ 0.167 each

**Stump 1** learns a simple rule from the data:  
- high income → approved  
- low income → denied  
- medium income → approved (majority of medium rows)

**Results:**

| Row | Actual | Predicted | Correct? |
|---|---|---|---|
| 0 | approved | approved | ✓ |
| 1 | approved | approved | ✓ |
| 2 | denied | denied | ✓ |
| 3 | denied | denied | ✓ |
| 4 | approved | approved | ✓ |
| 5 | **denied** | **approved** | ✗ ← wrong |

**Weighted error** = weight of wrong rows = 0.167  
**Alpha** = 0.805 (this stump was pretty good, so it gets a decent vote)

**New weights:**  
Row 5 (wrong) gets weight **× e^(+alpha)** → goes up  
All others (correct) get weight **× e^(-alpha)** → go down  
Then normalise so all weights sum to 1.

| Row | Old weight | New weight |
|---|---|---|
| 0–4 | 0.167 | **0.10** |
| 5 | 0.167 | **0.50** ← now 3× more important |

---
### Traced by hand — Round 2

**Stump 2** now sees row 5 as 3× more important.  
It changes its rule for medium income:  
- medium income → **denied** (to get row 5 right)

Now row 4 (medium, approved) becomes wrong instead.

**Each stump corrects the previous one's blind spot.**  
Final answer = weighted vote across all stumps using their alphas.

In [2]:
# AdaBoost — traced step by step
# Labels as numbers: approved=+1, denied=-1
labels  = [1, 1, -1, -1, 1, -1]
income  = ['high', 'high', 'low', 'low', 'medium', 'medium']
n       = 6
weights = [1/n] * n

# Two stumps defined by hand (matching the trace above)
stumps = [
    {'high': 1, 'low': -1, 'medium':  1},   # Round 1: medium → approved
    {'high': 1, 'low': -1, 'medium': -1},   # Round 2: medium → denied
]

print(f'  {"Row":<5} {"Income":<10} {"Actual":<12} {"Weight"}')
print('  ' + '-'*40)
for i in range(n):
    label_str = 'approved' if labels[i]==1 else 'denied'
    print(f'  {i:<5} {income[i]:<10} {label_str:<12} {weights[i]:.4f}')

alphas = []
for round_num, stump in enumerate(stumps):
    preds    = [stump[income[i]] for i in range(n)]
    wrong    = [i for i in range(n) if preds[i] != labels[i]]
    err      = sum(weights[i] for i in wrong)
    err      = max(min(err, 1-1e-10), 1e-10)
    alpha    = 0.5 * math.log((1 - err) / err)
    alphas.append(alpha)

    # Update weights
    new_w = [weights[i] * math.exp(-alpha * labels[i] * preds[i]) for i in range(n)]
    total = sum(new_w)
    weights = [w/total for w in new_w]

    print(f'\n--- Round {round_num+1} ---')
    print(f'  Stump rule : {stump}')
    print(f'  Wrong rows : {wrong}')
    print(f'  Error      : {err:.4f}')
    print(f'  Alpha      : {alpha:.4f}  ← trust this stump this much in the final vote')
    print(f'  New weights: {[round(w,4) for w in weights]}')
    print(f'  Row 5 weight: {weights[5]:.4f}  Row 4 weight: {weights[4]:.4f}')

  Row   Income     Actual       Weight
  ----------------------------------------
  0     high       approved     0.1667
  1     high       approved     0.1667
  2     low        denied       0.1667
  3     low        denied       0.1667
  4     medium     approved     0.1667
  5     medium     denied       0.1667

--- Round 1 ---
  Stump rule : {'high': 1, 'low': -1, 'medium': 1}
  Wrong rows : [5]
  Error      : 0.1667
  Alpha      : 0.8047  ← trust this stump this much in the final vote
  New weights: [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]
  Row 5 weight: 0.5000  Row 4 weight: 0.1000

--- Round 2 ---
  Stump rule : {'high': 1, 'low': -1, 'medium': -1}
  Wrong rows : [4]
  Error      : 0.1000
  Alpha      : 1.0986  ← trust this stump this much in the final vote
  New weights: [0.0556, 0.0556, 0.0556, 0.0556, 0.5, 0.2778]
  Row 5 weight: 0.2778  Row 4 weight: 0.5000


In [3]:
# Final AdaBoost prediction = weighted vote across all stumps
print('Final AdaBoost predictions (weighted vote):')
print(f'  Alpha values: {[round(a,4) for a in alphas]}')
print()
print(f'  {"Row":<5} {"Income":<10} {"Actual":<12} {"Score":<10} {"Predicted":<12} {"Correct?"}')
print('  ' + '-'*60)

actuals_str = ['approved','approved','denied','denied','approved','denied']
for i in range(n):
    score = sum(alphas[r] * stumps[r][income[i]] for r in range(len(stumps)))
    pred  = 'approved' if score >= 0 else 'denied'
    ok    = '✓' if pred == actuals_str[i] else '✗'
    print(f'  {i:<5} {income[i]:<10} {actuals_str[i]:<12} {score:<10.4f} {pred:<12} {ok}')

print()
print('Key insight: row 5 is denied because stump 2 (alpha=1.099) outweighs stump 1 (alpha=0.805).')
print('Higher alpha = louder voice in the final vote.')

Final AdaBoost predictions (weighted vote):
  Alpha values: [0.8047, 1.0986]

  Row   Income     Actual       Score      Predicted    Correct?
  ------------------------------------------------------------
  0     high       approved     1.9033     approved     ✓
  1     high       approved     1.9033     approved     ✓
  2     low        denied       -1.9033    denied       ✓
  3     low        denied       -1.9033    denied       ✓
  4     medium     approved     -0.2939    denied       ✗
  5     medium     denied       -0.2939    denied       ✓

Key insight: row 5 is denied because stump 2 (alpha=1.099) outweighs stump 1 (alpha=0.805).
Higher alpha = louder voice in the final vote.


---
# Part 2 — Gradient Boosting
## "Each tree fixes what the previous trees got wrong — numerically"

### The idea

Instead of reweighting rows, Gradient Boosting works with **residuals** — the gap between the actual value and the current prediction.

- Labels become numbers: approved = 1, denied = 0
- Start with a single number prediction for everyone (the mean)
- Tree 1 predicts the residuals (errors)
- Tree 2 predicts the residuals of (original − tree 1's correction)
- Each step nudges predictions closer to the truth

Think of it as **gradually correcting** rather than suddenly switching.

---
### Traced by hand

**Step 0 — Initial prediction:** mean of y = (1+1+0+0+1+0)/6 = **0.5** for everyone

**Step 1 — Residuals** = actual − predicted:

| Row | Income | Actual (y) | Predicted | Residual |
|---|---|---|---|---|
| 0 | high | 1 | 0.5 | +0.5 |
| 1 | high | 1 | 0.5 | +0.5 |
| 2 | low | 0 | 0.5 | −0.5 |
| 3 | low | 0 | 0.5 | −0.5 |
| 4 | medium | 1 | 0.5 | +0.5 |
| 5 | medium | 0 | 0.5 | −0.5 |

**Tree 1** learns to predict these residuals:
- high → mean residual of rows 0,1 = **+0.5**
- low → mean residual of rows 2,3 = **−0.5**
- medium → mean residual of rows 4,5 = **0.0** (one +0.5, one −0.5, average = 0)

**Update:** new prediction = 0.5 + (learning_rate × tree1_output)  
Using learning_rate = 0.5:

| Row | Income | Old pred | Tree 1 output | New pred |
|---|---|---|---|---|
| 0 | high | 0.5 | +0.5 | **0.75** |
| 1 | high | 0.5 | +0.5 | **0.75** |
| 2 | low | 0.5 | −0.5 | **0.25** |
| 3 | low | 0.5 | −0.5 | **0.25** |
| 4 | medium | 0.5 | 0.0 | **0.50** |
| 5 | medium | 0.5 | 0.0 | **0.50** |

**Step 2 — New residuals** are smaller for high and low rows — those are already well predicted.  
Medium rows still have residuals of ±0.5 — the next tree will focus there.

Over many rounds, all residuals shrink toward zero.

In [4]:
# Gradient Boosting — step by step
y      = [1, 1, 0, 0, 1, 0]   # approved=1, denied=0
income = ['high','high','low','low','medium','medium']
lr     = 0.5
n      = 6

F = [sum(y)/n] * n   # initial prediction = mean
print(f'Initial prediction for all rows: {F[0]}')
print()

for round_num in range(3):
    residuals = [y[i] - F[i] for i in range(n)]

    # Group residuals by income and take the mean as leaf value
    groups = {}
    for i in range(n):
        groups.setdefault(income[i], []).append(residuals[i])
    leaf = {k: sum(v)/len(v) for k, v in groups.items()}

    # Update predictions
    F = [F[i] + lr * leaf[income[i]] for i in range(n)]

    print(f'--- Round {round_num+1} ---')
    print(f'  Residuals : {[round(r,4) for r in residuals]}')
    print(f'  Leaf values: high={leaf["high"]:.3f}  low={leaf["low"]:.3f}  medium={leaf["medium"]:.3f}')
    print(f'  Predictions: {[round(f,4) for f in F]}')
    print()

print('Final predictions → threshold at 0.5:')
actuals_str = ['approved','approved','denied','denied','approved','denied']
print(f'  {"Row":<5} {"Income":<10} {"Actual":<12} {"Score":<8} {"Predicted":<12} {"Correct?"}')
print('  ' + '-'*55)
for i in range(n):
    pred = 'approved' if F[i] >= 0.5 else 'denied'
    ok   = '✓' if pred == actuals_str[i] else '✗'
    print(f'  {i:<5} {income[i]:<10} {actuals_str[i]:<12} {F[i]:<8.4f} {pred:<12} {ok}')

Initial prediction for all rows: 0.5

--- Round 1 ---
  Residuals : [0.5, 0.5, -0.5, -0.5, 0.5, -0.5]
  Leaf values: high=0.500  low=-0.500  medium=0.000
  Predictions: [0.75, 0.75, 0.25, 0.25, 0.5, 0.5]

--- Round 2 ---
  Residuals : [0.25, 0.25, -0.25, -0.25, 0.5, -0.5]
  Leaf values: high=0.250  low=-0.250  medium=0.000
  Predictions: [0.875, 0.875, 0.125, 0.125, 0.5, 0.5]

--- Round 3 ---
  Residuals : [0.125, 0.125, -0.125, -0.125, 0.5, -0.5]
  Leaf values: high=0.125  low=-0.125  medium=0.000
  Predictions: [0.9375, 0.9375, 0.0625, 0.0625, 0.5, 0.5]

Final predictions → threshold at 0.5:
  Row   Income     Actual       Score    Predicted    Correct?
  -------------------------------------------------------
  0     high       approved     0.9375   approved     ✓
  1     high       approved     0.9375   approved     ✓
  2     low        denied       0.0625   denied       ✓
  3     low        denied       0.0625   denied       ✓
  4     medium     approved     0.5000   approved     

---
# Part 3 — XGBoost
## "Gradient Boosting, but smarter about leaf values"

### The idea

XGBoost does the same thing as Gradient Boosting — sequential trees on residuals — but with one important addition:

> **Regularisation** — it deliberately makes the leaf values smaller to avoid overconfident predictions.

In plain Gradient Boosting, the leaf value = **mean of residuals**.  
In XGBoost, the leaf value = **mean of residuals, shrunk by a penalty λ (lambda)**:

$$\text{leaf} = \frac{\text{sum of residuals}}{\text{number of rows in leaf} + \lambda}$$

When λ = 0 → same as Gradient Boosting (no shrinkage)  
When λ is large → leaf values are forced toward zero → model is more conservative

---
### Traced by hand

**Same Round 1 residuals as Gradient Boosting:** [+0.5, +0.5, −0.5, −0.5, +0.5, −0.5]

**Leaf values with λ = 1:**

| Group | Sum of residuals | n in leaf | λ | Leaf value |
|---|---|---|---|---|
| high | 0.5+0.5 = 1.0 | 2 | 1 | 1.0 / (2+1) = **0.333** |
| low | −0.5+(−0.5) = −1.0 | 2 | 1 | −1.0 / (2+1) = **−0.333** |
| medium | 0.5+(−0.5) = 0.0 | 2 | 1 | 0.0 / (2+1) = **0.000** |

Compare to plain GB leaf values: high=0.5, low=−0.5, medium=0.0  
XGBoost's leaves are **smaller** (0.333 vs 0.5) — it takes smaller, safer steps.

This is why XGBoost generalises better — it never overcorrects on training data.

In [5]:
# XGBoost leaf value comparison across different lambda values
residuals_high   = [0.5, 0.5]
residuals_low    = [-0.5, -0.5]
residuals_medium = [0.5, -0.5]

def xgb_leaf(residuals, lam):
    return sum(residuals) / (len(residuals) + lam)

print('XGBoost leaf values for different lambda (Round 1):')
print(f'  {"Lambda":<10} {"high leaf":>12} {"low leaf":>12} {"medium leaf":>12}')
print('  ' + '-'*48)
for lam in [0, 0.5, 1, 2, 5]:
    h = xgb_leaf(residuals_high, lam)
    l = xgb_leaf(residuals_low, lam)
    m = xgb_leaf(residuals_medium, lam)
    note = '  ← same as plain GB' if lam == 0 else ''
    print(f'  {lam:<10} {h:>12.4f} {l:>12.4f} {m:>12.4f}{note}')

print()
print('As lambda increases → leaf values shrink toward 0 → more conservative model.')

XGBoost leaf values for different lambda (Round 1):
  Lambda        high leaf     low leaf  medium leaf
  ------------------------------------------------
  0                0.5000      -0.5000       0.0000  ← same as plain GB
  0.5              0.4000      -0.4000       0.0000
  1                0.3333      -0.3333       0.0000
  2                0.2500      -0.2500       0.0000
  5                0.1429      -0.1429       0.0000

As lambda increases → leaf values shrink toward 0 → more conservative model.


---
# Summary — The Three Boosting Algorithms

| | AdaBoost | Gradient Boosting | XGBoost |
|---|---|---|---|
| **Core trick** | Reweight misclassified rows | Fit residuals (errors) numerically | Fit residuals + regularise leaf values |
| **What changes each round** | Sample weights | Residuals | Residuals (with shrinkage) |
| **Final prediction** | Weighted vote of stumps | Sum all tree outputs | Sum all tree outputs |
| **Key parameter** | Number of stumps | Learning rate | Learning rate + Lambda |
| **Risk** | Sensitive to outliers (they get very high weight) | Can overfit if learning rate is too high | Less likely to overfit due to regularisation |

---

### The one thing to remember about each

**AdaBoost** → *"I got that wrong, let me pay extra attention to it next time"*  
**Gradient Boosting** → *"Here is the gap between my answer and the truth — let me fill it"*  
**XGBoost** → *"Same as GB, but I'll take a smaller, safer step each time"*